Below is **Notebook 01 only**, ready to copy cell by cell. It uses `mp_api.client.MPRester`; MP’s official docs describe `MPRester` as the client for the `mp-api` package, and Materials Project support has pointed users to `mpr.materials.insertion_electrodes.search()` for Battery Explorer insertion-electrode data. The code still **does not assume that exact route or field names are always available**; it resolves endpoints and fields dynamically and logs fallbacks. ([Materials Project Documentation][1])

-

# `01_multion_data_extraction.ipynb`

## Cell 1 — Notebook identity, imports, output folders

In [ ]:
# ============================================================
# Notebook 01: Multi-ion MP insertion-electrode extraction
# CMT Path A: Leakage-audited multi-ion computed electrode benchmark
# ============================================================
#
# Purpose:
#   Build and audit a Li-Na-K Materials Project insertion-electrode dataset.
#
# Strict exclusions in this notebook:
#   - No ML
#   - No descriptor generation
#   - No ranking
#   - No CDE matching
#   - No criticality filtering
#   - No manuscript writing
#
# Outputs:
#   Clean-room repository sections under data/ and provenance/.
#
# Security:
#   - Never save or print Materials Project API key.
#   - Use MP_API_KEY environment variable or secure input prompt.
# ============================================================

from __future__ import annotations

import os
import sys
import re
import json
import math
import inspect
import traceback
import getpass
import platform
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from itertools import combinations
from dataclasses import is_dataclass, asdict

import numpy as np
import pandas as pd

try:
    import importlib.metadata as importlib_metadata
except Exception:
    import importlib_metadata

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = lambda x, **kwargs: x

# -
# Output directories
# -
def _locate_repository_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for root in [candidate, *candidate.parents]:
        if (
            (root / "notebooks").is_dir()
            and (root / "data").is_dir()
            and (root / "results").is_dir()
            and (root / "provenance").is_dir()
        ):
            return root
    raise FileNotFoundError("Could not locate the clean-room repository root.")


REPOSITORY_ROOT = _locate_repository_root()
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from software.cmt_repository_paths import artifact_namespace, runtime_cache_root

BASE_DIR = artifact_namespace("01", REPOSITORY_ROOT)
RAW_DIR = BASE_DIR / "raw"
PROCESSED_DIR = BASE_DIR / "processed"
AUDIT_DIR = BASE_DIR / "audit"
METADATA_DIR = BASE_DIR / "metadata"
LOG_DIR = BASE_DIR / "logs"

for d in [RAW_DIR, PROCESSED_DIR, AUDIT_DIR, METADATA_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP_UTC = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
WORKING_IONS = ["Li", "Na", "K"]

LOG_ROWS = []

def log_event(stage: str, level: str, message: str, extra: dict | None = None):
    """Append a structured log row. Does not print sensitive data."""
    row = {
        "timestamp_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "stage": stage,
        "level": level,
        "message": message,
        "extra_json": json.dumps(extra or {}, default=str),
    }
    LOG_ROWS.append(row)
    if level.upper() in {"WARNING", "ERROR"}:
        print(f"[{level.upper()}] {stage}: {message}")

def save_event_log():
    pd.DataFrame(LOG_ROWS).to_csv(LOG_DIR / "01_event_log.csv", index=False)

log_event("init", "INFO", "Notebook 01 initialized.", {"base_dir": str(BASE_DIR)})
print(f"Notebook 01 output directory: {BASE_DIR}")


-

## Cell 2 — Serialization, flattening, and safe utility functions

In [ ]:
# ============================================================
# Robust serialization and extraction utilities
# ============================================================

def package_version(package_name: str) -> str:
    try:
        return importlib_metadata.version(package_name)
    except Exception:
        return "not_installed_or_unknown"

def is_missing_like(x) -> bool:
    if x is None:
        return True
    if isinstance(x, float) and math.isnan(x):
        return True
    if isinstance(x, (list, tuple, set, dict)) and len(x) == 0:
        return True
    if isinstance(x, str) and x.strip() == "":
        return True
    return False

def to_builtin(obj, max_depth: int = 25):
    """
    Convert pydantic models, pymatgen objects, numpy scalars, dataclasses,
    dict/list objects, etc. into JSON-serializable built-ins.

    This is intentionally defensive because MP API document object types
    can change between mp-api versions.
    """
    if max_depth <= 0:
        return str(obj)

    if obj is None or isinstance(obj, (str, int, bool)):
        return obj

    if isinstance(obj, float):
        return obj if math.isfinite(obj) else None

    if isinstance(obj, (np.integer, np.floating, np.bool_)):
        val = obj.item()
        if isinstance(val, float) and not math.isfinite(val):
            return None
        return val

    if isinstance(obj, Path):
        return str(obj)

    if is_dataclass(obj):
        try:
            return to_builtin(asdict(obj), max_depth=max_depth - 1)
        except Exception:
            pass

    if hasattr(obj, "model_dump"):
        for kwargs in [{"mode": "json"}, {}]:
            try:
                return to_builtin(obj.model_dump(**kwargs), max_depth=max_depth - 1)
            except Exception:
                pass

    if hasattr(obj, "dict"):
        try:
            return to_builtin(obj.dict(), max_depth=max_depth - 1)
        except Exception:
            pass

    if hasattr(obj, "as_dict"):
        try:
            return to_builtin(obj.as_dict(), max_depth=max_depth - 1)
        except Exception:
            pass

    if isinstance(obj, dict):
        out = {}
        for k, v in obj.items():
            try:
                key = str(k)
            except Exception:
                key = repr(k)
            out[key] = to_builtin(v, max_depth=max_depth - 1)
        return out

    if isinstance(obj, (list, tuple, set)):
        return [to_builtin(v, max_depth=max_depth - 1) for v in obj]

    # Last resort: string representation
    return str(obj)

def write_json_safe(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(to_builtin(obj), f, ensure_ascii=False, indent=2)

def docs_to_flat_dataframe(docs_builtin: list[dict]) -> pd.DataFrame:
    if not docs_builtin:
        return pd.DataFrame()
    try:
        return pd.json_normalize(docs_builtin, sep=".")
    except Exception as exc:
        log_event("flatten", "WARNING", "pd.json_normalize failed; using shallow DataFrame.", {"error": str(exc)})
        return pd.DataFrame(docs_builtin)

def deep_find_values_exact_key(obj, wanted_key: str, max_depth: int = 25):
    """Find values for exact key name anywhere in nested object."""
    values = []
    if max_depth <= 0:
        return values

    if isinstance(obj, dict):
        for k, v in obj.items():
            if str(k) == wanted_key:
                values.append(v)
            values.extend(deep_find_values_exact_key(v, wanted_key, max_depth=max_depth - 1))
    elif isinstance(obj, list):
        for item in obj:
            values.extend(deep_find_values_exact_key(item, wanted_key, max_depth=max_depth - 1))
    return values

def get_by_dotted_path(obj, dotted_path: str):
    """Get value from nested dict using dotted path. Returns None if unavailable."""
    cur = obj
    for part in dotted_path.split("."):
        if isinstance(cur, dict) and part in cur:
            cur = cur[part]
        else:
            return None
    return cur

def get_any_alias(doc: dict, aliases: list[str], default=None):
    """
    Extract first non-missing value from direct key, dotted path, or exact recursive key.
    Does not assume a fixed MP schema.
    """
    if not isinstance(doc, dict):
        return default

    # 1. Direct key
    for a in aliases:
        if a in doc and not is_missing_like(doc[a]):
            return doc[a]

    # 2. Dotted path
    for a in aliases:
        if "." in a:
            val = get_by_dotted_path(doc, a)
            if not is_missing_like(val):
                return val

    # 3. Recursive exact key fallback
    for a in aliases:
        vals = deep_find_values_exact_key(doc, a)
        for val in vals:
            if not is_missing_like(val):
                return val

    return default

MP_ID_REGEX = re.compile(r"\bmp-\d+\b")

def extract_mp_ids_from_object(obj) -> list[str]:
    """Extract Materials Project IDs from any nested object by regex on JSON/string representation."""
    try:
        text = json.dumps(to_builtin(obj), ensure_ascii=False)
    except Exception:
        text = str(obj)
    return sorted(set(MP_ID_REGEX.findall(text)))

def stable_hash_text(text: str, n: int = 12) -> str:
    return hashlib.sha256(text.encode("utf-8", errors="ignore")).hexdigest()[:n]

def safe_float(x):
    if is_missing_like(x):
        return np.nan
    try:
        return float(x)
    except Exception:
        return np.nan

def safe_str(x):
    if is_missing_like(x):
        return ""
    if isinstance(x, (list, tuple, set)):
        return "|".join([safe_str(v) for v in x if not is_missing_like(v)])
    if isinstance(x, dict):
        return json.dumps(to_builtin(x), sort_keys=True, ensure_ascii=False)
    return str(x)

def chunked(seq, size: int):
    seq = list(seq)
    for i in range(0, len(seq), size):
        yield seq[i:i + size]

save_event_log()


-

## Cell 3 — Save software environment metadata

In [ ]:
# ============================================================
# Software environment metadata
# ============================================================

software_environment = {
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
    "python_version": sys.version,
    "platform": platform.platform(),
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "mp_api_version": package_version("mp-api"),
    "pymatgen_version": package_version("pymatgen"),
    "tqdm_version": package_version("tqdm"),
    "notebook_purpose": "Li-Na-K Materials Project insertion-electrode extraction and feasibility audit",
}

write_json_safe(software_environment, METADATA_DIR / "01_software_environment.json")
print(json.dumps(software_environment, indent=2))


-

## Cell 4 — Materials Project API API authentication

In [ ]:
# ============================================================
# Materials Project API key handling
# ============================================================
#
# Priority:
#   1. Environment variable MP_API_KEY
#   2. Environment variable PMG_MAPI_KEY
#   3. Secure input prompt
#
# The key is never printed and never saved.
# ============================================================

try:
    from mp_api.client import MPRester
except Exception as exc:
    raise ImportError(
        "mp-api is required. Install with: pip install mp-api pymatgen"
    ) from exc

MP_API_KEY = os.environ.get("MP_API_KEY", None)

if not MP_API_KEY:
    # Some pymatgen/MP setups use this older environment variable.
    MP_API_KEY = os.environ.get("PMG_MAPI_KEY", None)

if not MP_API_KEY:
    MP_API_KEY = getpass.getpass("Enter Materials Project API key securely: ")

if not MP_API_KEY or not MP_API_KEY.strip():
    raise RuntimeError("No Materials Project API key provided. Set MP_API_KEY or use secure prompt.")

log_event("api_key", "INFO", "API key loaded securely. Key was not printed or saved.")
save_event_log()

print("Materials Project API key loaded securely. It was not printed or saved.")


-

## Cell 5 — Connect to MP and resolve endpoint routes dynamically

In [ ]:
# ============================================================
# Connect to Materials Project and resolve endpoint routes
# ============================================================

def resolve_nested_attr(root, dotted_path: str):
    cur = root
    for part in dotted_path.split("."):
        if not hasattr(cur, part):
            return None
        cur = getattr(cur, part)
    return cur

def resolve_rester(mpr, candidate_paths: list[str], required_method: str = "search"):
    """
    Try multiple possible API route paths.
    Returns: (label, rester_object)
    """
    tried = []
    for path in candidate_paths:
        try:
            obj = resolve_nested_attr(mpr, path)
            tried.append(path)
            if obj is not None and hasattr(obj, required_method):
                return path, obj, tried
        except Exception as exc:
            log_event("resolve_rester", "WARNING", f"Failed to inspect route {path}.", {"error": str(exc)})
    return None, None, tried

def safe_get_database_version(mpr):
    """
    Try multiple possible ways to retrieve database version.
    If unavailable, return unknown rather than failing.
    """
    for name in ["get_database_version", "get_db_version"]:
        if hasattr(mpr, name):
            try:
                return getattr(mpr, name)()
            except Exception as exc:
                log_event("database_version", "WARNING", f"{name} failed.", {"error": str(exc)})

    for attr in ["database_version", "db_version", "_db_version"]:
        if hasattr(mpr, attr):
            try:
                val = getattr(mpr, attr)
                if not callable(val):
                    return val
            except Exception:
                pass

    return "unknown_or_unavailable"

mpr = MPRester(MP_API_KEY)

MP_DATABASE_VERSION = safe_get_database_version(mpr)

ELECTRODE_ROUTE_CANDIDATES = [
    "materials.insertion_electrodes",
    "insertion_electrodes",
    "materials.electrodes",
    "electrodes",
]

SUMMARY_ROUTE_CANDIDATES = [
    "materials.summary",
    "summary",
]

electrode_route_label, electrode_rester, electrode_routes_tried = resolve_rester(
    mpr, ELECTRODE_ROUTE_CANDIDATES, required_method="search"
)

summary_route_label, summary_rester, summary_routes_tried = resolve_rester(
    mpr, SUMMARY_ROUTE_CANDIDATES, required_method="search"
)

if electrode_rester is None:
    write_json_safe(
        {
            "status": "failed",
            "routes_tried": electrode_routes_tried,
            "message": "Could not resolve insertion-electrode endpoint.",
        },
        METADATA_DIR / "01_mp_connection_metadata.json",
    )
    save_event_log()
    raise RuntimeError(
        "Could not resolve a Materials Project insertion-electrode endpoint. "
        "Check mp-api version and MP endpoint availability."
    )

connection_metadata = {
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
    "mp_database_version": to_builtin(MP_DATABASE_VERSION),
    "electrode_route_selected": electrode_route_label,
    "electrode_routes_tried": electrode_routes_tried,
    "summary_route_selected": summary_route_label,
    "summary_routes_tried": summary_routes_tried,
    "summary_route_available": summary_rester is not None,
    "api_key_saved": False,
    "api_key_printed": False,
}

write_json_safe(connection_metadata, METADATA_DIR / "01_mp_connection_metadata.json")

log_event(
    "connection",
    "INFO",
    "MP connection established and endpoint routes resolved.",
    {
        "electrode_route_selected": electrode_route_label,
        "summary_route_selected": summary_route_label,
        "mp_database_version": str(MP_DATABASE_VERSION),
    },
)

save_event_log()

print("MP connection established.")
print(f"Selected electrode route: {electrode_route_label}")
print(f"Selected summary route: {summary_route_label}")
print(f"MP database version: {MP_DATABASE_VERSION}")


-

## Cell 6 — Discover available fields dynamically

In [ ]:
# ============================================================
# Dynamic field discovery
# ============================================================

def discover_available_fields(rester) -> list[str]:
    """
    Discover available fields from multiple possible mp-api rester attributes.
    Returns a sorted unique list. Empty list means unavailable.
    """
    fields = []

    # Common mp-api property/method
    for attr in ["available_fields", "_available_fields"]:
        try:
            if hasattr(rester, attr):
                val = getattr(rester, attr)
                if callable(val):
                    val = val()
                if val:
                    fields.extend([str(x) for x in list(val)])
        except Exception as exc:
            log_event("field_discovery", "WARNING", f"Could not read {attr}.", {"error": str(exc)})

    # Pydantic document model fields
    for attr in ["document_model", "_document_model"]:
        try:
            if hasattr(rester, attr):
                model = getattr(rester, attr)
                if hasattr(model, "model_fields"):
                    fields.extend([str(x) for x in model.model_fields.keys()])
                elif hasattr(model, "__fields__"):
                    fields.extend([str(x) for x in model.__fields__.keys()])
        except Exception as exc:
            log_event("field_discovery", "WARNING", f"Could not inspect {attr}.", {"error": str(exc)})

    return sorted(set(fields))

def search_signature_report(rester) -> dict:
    try:
        sig = inspect.signature(rester.search)
        params = sig.parameters
        return {
            "signature": str(sig),
            "parameters": list(params.keys()),
            "accepts_var_kwargs": any(p.kind == inspect.Parameter.VAR_KEYWORD for p in params.values()),
        }
    except Exception as exc:
        return {
            "signature": "unavailable",
            "parameters": [],
            "accepts_var_kwargs": False,
            "error": str(exc),
        }

available_electrode_fields = discover_available_fields(electrode_rester)
available_summary_fields = discover_available_fields(summary_rester) if summary_rester is not None else []

electrode_search_signature = search_signature_report(electrode_rester)
summary_search_signature = search_signature_report(summary_rester) if summary_rester is not None else {}

pd.DataFrame({"field": available_electrode_fields}).to_csv(
    AUDIT_DIR / "01_available_insertion_electrode_fields.csv",
    index=False,
)

pd.DataFrame({"field": available_summary_fields}).to_csv(
    AUDIT_DIR / "01_available_summary_fields.csv",
    index=False,
)

write_json_safe(
    {
        "electrode_route": electrode_route_label,
        "n_available_electrode_fields": len(available_electrode_fields),
        "available_electrode_fields": available_electrode_fields,
        "electrode_search_signature": electrode_search_signature,
        "summary_route": summary_route_label,
        "n_available_summary_fields": len(available_summary_fields),
        "available_summary_fields": available_summary_fields,
        "summary_search_signature": summary_search_signature,
    },
    METADATA_DIR / "01_field_discovery_metadata.json",
)

print(f"Available electrode fields discovered: {len(available_electrode_fields)}")
print(f"Available summary fields discovered: {len(available_summary_fields)}")
print(f"Electrode search signature: {electrode_search_signature.get('signature')}")
save_event_log()


-

## Cell 7 — Robust insertion-electrode search helpers

In [ ]:
# ============================================================
# Robust MP insertion-electrode search helpers
# ============================================================

# These are candidate field aliases only. The notebook does not assume they exist.
# If dynamic field discovery is available, only existing fields are requested.
PREFERRED_ELECTRODE_FIELD_CANDIDATES = [
    "battery_id",
    "id",
    "task_id",
    "material_id",
    "working_ion",
    "battery_formula",
    "formula_charge",
    "formula_discharge",
    "framework_formula",
    "average_voltage",
    "capacity_grav",
    "capacity_vol",
    "energy_grav",
    "energy_vol",
    "max_delta_volume",
    "stability_charge",
    "stability_discharge",
    "fracA_charge",
    "fracA_discharge",
    "num_steps",
    "max_voltage_step",
    "id_charge",
    "id_discharge",
    "material_ids",
    "chemsys",
    "elements",
    "battery_type",
    "thermo_type",
    "last_updated",
    "host_structure",
    "electrode_object",
    "warnings",
]

def field_strategy_list(available_fields: list[str]) -> list[tuple[str, list[str] | None]]:
    """
    Return ordered field strategies.
    1. All discovered fields, if available.
    2. Preferred fields that are confirmed available, if any.
    3. No fields argument.
    """
    strategies = []

    if available_fields:
        strategies.append(("all_discovered_fields", sorted(set(available_fields))))

        preferred_available = [f for f in PREFERRED_ELECTRODE_FIELD_CANDIDATES if f in set(available_fields)]
        if preferred_available:
            strategies.append(("preferred_available_fields", preferred_available))

    # If no field discovery exists, try preferred names once, then no fields.
    if not available_fields:
        strategies.append(("preferred_candidate_fields_unverified", PREFERRED_ELECTRODE_FIELD_CANDIDATES))

    strategies.append(("no_fields_argument", None))
    return strategies

def accepts_kwarg(rester, kwarg_name: str) -> bool:
    report = search_signature_report(rester)
    params = report.get("parameters", [])
    return kwarg_name in params or report.get("accepts_var_kwargs", False)

def call_search(rester, kwargs: dict, stage: str):
    """
    Safe wrapper around rester.search(**kwargs).
    Returns (docs_list, error_string).
    """
    try:
        docs = rester.search(**kwargs)
        docs = list(docs)
        return docs, None
    except Exception as exc:
        err = f"{type(exc).__name__}: {str(exc)}"
        log_event(stage, "WARNING", "Search attempt failed.", {"kwargs_keys": list(kwargs.keys()), "error": err})
        return None, err

def infer_working_ion_from_doc(doc_builtin: dict) -> str | None:
    """
    Infer working ion from explicit field if available, otherwise from battery formula.
    This is only a fallback for client-side filtering.
    """
    explicit = get_any_alias(doc_builtin, ["working_ion", "workingIon", "ion", "workingIonSymbol"], default=None)
    if not is_missing_like(explicit):
        s = safe_str(explicit).strip()
        for ion in WORKING_IONS:
            if s == ion or s.lower() == ion.lower():
                return ion

    formula_candidates = [
        get_any_alias(doc_builtin, ["battery_formula", "batteryFormula"], default=None),
        get_any_alias(doc_builtin, ["formula_discharge", "formulaDischarge"], default=None),
        get_any_alias(doc_builtin, ["formula_charge", "formulaCharge"], default=None),
        get_any_alias(doc_builtin, ["framework_formula", "frameworkFormula"], default=None),
    ]

    for f in formula_candidates:
        fs = safe_str(f).strip()
        # Common MP battery formula style: Li0-1CoO2, Na0-1MnO2, K0-1...
        m = re.match(r"^(Li|Na|K)(?=[0-9\.\-])", fs)
        if m:
            return m.group(1)

    return None

ALL_ELECTRODE_DOCS_CACHE = {}

def get_all_electrode_docs_for_field_strategy(rester, field_label: str, fields: list[str] | None):
    """
    Search all insertion-electrode docs for a field strategy.
    Cached to avoid repeated all-endpoint downloads.
    """
    cache_key = field_label
    if cache_key in ALL_ELECTRODE_DOCS_CACHE:
        return ALL_ELECTRODE_DOCS_CACHE[cache_key]

    kwargs = {}
    if fields is not None and accepts_kwarg(rester, "fields"):
        kwargs["fields"] = fields
    elif fields is not None:
        # Some mp-api versions expose **kwargs but signature may not show fields.
        kwargs["fields"] = fields

    docs, err = call_search(rester, kwargs, stage=f"search_all_{field_label}")
    if docs is None:
        return None, err

    ALL_ELECTRODE_DOCS_CACHE[cache_key] = docs
    return docs, None

def search_insertion_electrodes_for_ion(rester, ion: str, available_fields: list[str]):
    """
    Extract insertion-electrode records for one working ion using robust fallback logic.

    Attempts:
      1. working_ion=ion
      2. working_ions=[ion]
      3. working_ion=[ion]
      4. all docs + client-side inference/filtering
    """
    strategies = field_strategy_list(available_fields)

    ion_kwarg_variants = [
        ("working_ion_string", {"working_ion": ion}),
        ("working_ions_list", {"working_ions": [ion]}),
        ("working_ion_list", {"working_ion": [ion]}),
    ]

    attempt_log = []

    for field_label, fields in strategies:
        for ion_label, ion_kwargs in ion_kwarg_variants:
            kwargs = {}

            if fields is not None:
                kwargs["fields"] = fields

            kwargs.update(ion_kwargs)

            docs, err = call_search(
                rester,
                kwargs,
                stage=f"search_{ion}_{field_label}_{ion_label}",
            )

            attempt_log.append({
                "ion": ion,
                "field_strategy": field_label,
                "ion_strategy": ion_label,
                "kwargs_keys": list(kwargs.keys()),
                "success": docs is not None,
                "n_docs": len(docs) if docs is not None else None,
                "error": err,
            })

            if docs is not None and len(docs) > 0:
                docs_builtin = [to_builtin(d) for d in docs]
                inferred = [infer_working_ion_from_doc(d) for d in docs_builtin]
                n_match = sum(x == ion for x in inferred)

                # If most records match the requested ion or inference is mostly unavailable, accept.
                n_inferred = sum(x is not None for x in inferred)
                if n_inferred == 0 or n_match >= max(1, 0.8 * n_inferred):
                    return docs, {
                        "ion": ion,
                        "mode": "direct_endpoint_query",
                        "field_strategy": field_label,
                        "ion_strategy": ion_label,
                        "n_docs": len(docs),
                        "n_inferred": n_inferred,
                        "n_inferred_matching_ion": n_match,
                        "attempt_log": attempt_log,
                    }

                # If direct query returned mixed docs, filter them.
                filtered_docs = [d for d, inferred_ion in zip(docs, inferred) if inferred_ion == ion]
                if len(filtered_docs) > 0:
                    log_event(
                        "ion_search",
                        "WARNING",
                        f"Direct query for {ion} returned mixed inferred ions; using filtered subset.",
                        {"original_n": len(docs), "filtered_n": len(filtered_docs)},
                    )
                    return filtered_docs, {
                        "ion": ion,
                        "mode": "direct_query_then_client_filter",
                        "field_strategy": field_label,
                        "ion_strategy": ion_label,
                        "n_docs": len(filtered_docs),
                        "original_n_docs": len(docs),
                        "attempt_log": attempt_log,
                    }

        # Fallback: all docs + client-side filtering for this field strategy
        all_docs, err = get_all_electrode_docs_for_field_strategy(rester, field_label, fields)
        attempt_log.append({
            "ion": ion,
            "field_strategy": field_label,
            "ion_strategy": "all_docs_client_filter",
            "success": all_docs is not None,
            "n_docs": len(all_docs) if all_docs is not None else None,
            "error": err,
        })

        if all_docs is not None and len(all_docs) > 0:
            all_builtin = [to_builtin(d) for d in all_docs]
            inferred = [infer_working_ion_from_doc(d) for d in all_builtin]
            filtered_docs = [d for d, inferred_ion in zip(all_docs, inferred) if inferred_ion == ion]

            if len(filtered_docs) > 0:
                return filtered_docs, {
                    "ion": ion,
                    "mode": "all_docs_client_side_filter",
                    "field_strategy": field_label,
                    "n_docs": len(filtered_docs),
                    "all_docs_n": len(all_docs),
                    "attempt_log": attempt_log,
                }

    return [], {
        "ion": ion,
        "mode": "failed_or_empty",
        "n_docs": 0,
        "attempt_log": attempt_log,
    }

save_event_log()
print("Search helpers ready.")


-

## Cell 8 — Extract Li, Na, K raw records and save raw JSON/CSV immediately

In [ ]:
# ============================================================
# Extract raw Li, Na, and K insertion-electrode records
# Save raw JSON and raw flattened CSV before any cleaning.
# ============================================================

raw_docs_by_ion = {}
raw_builtin_by_ion = {}
extraction_audit_rows = []

for ion in WORKING_IONS:
    print(f"\n- Extracting {ion}-ion insertion-electrode records -")

    docs, meta = search_insertion_electrodes_for_ion(
        electrode_rester,
        ion=ion,
        available_fields=available_electrode_fields,
    )

    docs_builtin = [to_builtin(d) for d in docs]

    raw_docs_by_ion[ion] = docs
    raw_builtin_by_ion[ion] = docs_builtin

    # Save raw JSON immediately
    raw_json_path = RAW_DIR / f"${CMT_RAW_DATA_DIR}/mp_{ion}_insertion_electrodes_raw.json"
    write_json_safe(
        {
            "ion": ion,
            "run_timestamp_utc": RUN_TIMESTAMP_UTC,
            "mp_database_version": to_builtin(MP_DATABASE_VERSION),
            "electrode_route_selected": electrode_route_label,
            "extraction_metadata": meta,
            "n_records": len(docs_builtin),
            "records": docs_builtin,
        },
        raw_json_path,
    )

    # Save flattened raw CSV immediately
    flat_df = docs_to_flat_dataframe(docs_builtin)
    flat_csv_path = RAW_DIR / f"mp_{ion}_insertion_electrodes_raw_flat.csv"
    flat_df.to_csv(flat_csv_path, index=False)

    extraction_audit_rows.append({
        "working_ion": ion,
        "n_records": len(docs_builtin),
        "raw_json_path": str(raw_json_path),
        "raw_flat_csv_path": str(flat_csv_path),
        "mode": meta.get("mode"),
        "field_strategy": meta.get("field_strategy"),
        "ion_strategy": meta.get("ion_strategy"),
        "all_docs_n": meta.get("all_docs_n"),
        "electrode_route_selected": electrode_route_label,
        "mp_database_version": safe_str(MP_DATABASE_VERSION),
    })

    # Save per-ion attempt log
    write_json_safe(meta, METADATA_DIR / f"01_extraction_attempts_{ion}.json")

    print(f"{ion}: extracted {len(docs_builtin)} records")
    log_event("raw_extraction", "INFO", f"Extracted records for {ion}.", {"n_records": len(docs_builtin)})

extraction_audit_df = pd.DataFrame(extraction_audit_rows)
extraction_audit_df.to_csv(AUDIT_DIR / "01_raw_extraction_audit.csv", index=False)

write_json_safe(
    {
        "run_timestamp_utc": RUN_TIMESTAMP_UTC,
        "working_ions": WORKING_IONS,
        "records_by_ion": {ion: len(raw_builtin_by_ion.get(ion, [])) for ion in WORKING_IONS},
        "electrode_route_selected": electrode_route_label,
        "mp_database_version": to_builtin(MP_DATABASE_VERSION),
        "raw_outputs_saved": True,
        "api_key_saved": False,
        "api_key_printed": False,
    },
    METADATA_DIR / "01_extraction_run_manifest.json",
)

display(extraction_audit_df)
save_event_log()


-

## Cell 9 — Build harmonized core dataset

In [ ]:
# ============================================================
# Harmonize raw records into a core schema
# ============================================================

try:
    from pymatgen.core import Composition, Element
    PYMATGEN_AVAILABLE = True
except Exception as exc:
    PYMATGEN_AVAILABLE = False
    log_event("pymatgen", "WARNING", "pymatgen Composition/Element unavailable; formula parsing will be limited.", {"error": str(exc)})

# Candidate aliases only. Missing aliases are allowed and logged through coverage audits.
FIELD_ALIASES = {
    "record_source_id": ["battery_id", "id", "task_id", "material_id", "electrode_id"],
    "working_ion": ["working_ion", "workingIon", "ion", "workingIonSymbol"],
    "battery_formula": ["battery_formula", "batteryFormula"],
    "formula_charge": ["formula_charge", "formulaCharge", "charged_formula", "chargedFormula"],
    "formula_discharge": ["formula_discharge", "formulaDischarge", "discharged_formula", "dischargedFormula"],
    "framework_formula": ["framework_formula", "frameworkFormula", "host_formula", "hostFormula"],
    "chemsys": ["chemsys", "chemical_system", "chemicalSystem"],
    "elements": ["elements"],
    "average_voltage": ["average_voltage", "averageVoltage", "voltage", "avg_voltage"],
    "capacity_grav": ["capacity_grav", "capacityGrav", "gravimetric_capacity", "capacity_gravimetric"],
    "capacity_vol": ["capacity_vol", "capacityVol", "volumetric_capacity", "capacity_volumetric"],
    "energy_grav": ["energy_grav", "energyGrav", "gravimetric_energy", "energy_gravimetric"],
    "energy_vol": ["energy_vol", "energyVol", "volumetric_energy", "energy_volumetric"],
    "max_delta_volume": ["max_delta_volume", "maxDeltaVolume", "max_volume_change", "volume_change"],
    "stability_charge": ["stability_charge", "stabilityCharge"],
    "stability_discharge": ["stability_discharge", "stabilityDischarge"],
    "fracA_charge": ["fracA_charge", "fracACharge"],
    "fracA_discharge": ["fracA_discharge", "fracADischarge"],
    "num_steps": ["num_steps", "numSteps", "n_steps", "steps"],
    "max_voltage_step": ["max_voltage_step", "maxVoltageStep"],
    "id_charge": ["id_charge", "idCharge", "charge_id", "charged_material_id"],
    "id_discharge": ["id_discharge", "idDischarge", "discharge_id", "discharged_material_id"],
    "material_ids": ["material_ids", "materialIds", "materials_ids", "material_id_list"],
    "battery_type": ["battery_type", "batteryType"],
    "thermo_type": ["thermo_type", "thermoType"],
    "last_updated": ["last_updated", "lastUpdated", "updated_at"],
}

def formula_to_reduced_formula(formula) -> str:
    if is_missing_like(formula):
        return ""
    fs = safe_str(formula)
    if not fs:
        return ""
    if not PYMATGEN_AVAILABLE:
        return fs
    try:
        return Composition(fs).reduced_formula
    except Exception:
        return fs

def formula_to_elements(formula) -> list[str]:
    if is_missing_like(formula):
        return []
    fs = safe_str(formula)
    if not fs or not PYMATGEN_AVAILABLE:
        return []
    try:
        comp = Composition(fs)
        return sorted([str(el) for el in comp.elements])
    except Exception:
        return []

def normalize_elements_value(elements_value, fallback_formula=None) -> list[str]:
    if not is_missing_like(elements_value):
        if isinstance(elements_value, list):
            out = []
            for x in elements_value:
                sx = safe_str(x)
                # Element objects sometimes stringify as "Element Na"
                m = re.search(r"\b([A-Z][a-z]?)\b", sx)
                if m:
                    out.append(m.group(1))
                elif sx:
                    out.append(sx)
            return sorted(set(out))
        s = safe_str(elements_value)
        candidates = re.findall(r"\b[A-Z][a-z]?\b", s)
        if candidates:
            return sorted(set(candidates))

    return formula_to_elements(fallback_formula)

def derive_chemsys(elements: list[str]) -> str:
    if not elements:
        return ""
    return "-".join(sorted(set(elements)))

def harmonize_one_doc(doc: dict, queried_ion: str, i: int) -> dict:
    row = {
        "record_index": f"{queried_ion}_{i:06d}",
        "source_queried_working_ion": queried_ion,
        "source_endpoint": electrode_route_label,
        "mp_database_version": safe_str(MP_DATABASE_VERSION),
    }

    for standard_field, aliases in FIELD_ALIASES.items():
        row[standard_field] = get_any_alias(doc, aliases, default=None)

    # If working_ion is unavailable, use query ion or inferred ion.
    inferred = infer_working_ion_from_doc(doc)
    if is_missing_like(row.get("working_ion")):
        row["working_ion"] = inferred or queried_ion

    # Normalize scalar/string fields
    for col in [
        "record_source_id",
        "working_ion",
        "battery_formula",
        "formula_charge",
        "formula_discharge",
        "framework_formula",
        "chemsys",
        "battery_type",
        "thermo_type",
        "last_updated",
    ]:
        row[col] = safe_str(row.get(col))

    # Numeric fields
    numeric_cols = [
        "average_voltage",
        "capacity_grav",
        "capacity_vol",
        "energy_grav",
        "energy_vol",
        "max_delta_volume",
        "stability_charge",
        "stability_discharge",
        "fracA_charge",
        "fracA_discharge",
        "num_steps",
        "max_voltage_step",
    ]
    for col in numeric_cols:
        row[col] = safe_float(row.get(col))

    # Worst-case stability
    sc = row.get("stability_charge", np.nan)
    sd = row.get("stability_discharge", np.nan)
    if np.isfinite(sc) and np.isfinite(sd):
        row["stability_worst"] = max(sc, sd)
    elif np.isfinite(sc):
        row["stability_worst"] = sc
    elif np.isfinite(sd):
        row["stability_worst"] = sd
    else:
        row["stability_worst"] = np.nan

    # Material IDs
    row["id_charge"] = safe_str(row.get("id_charge"))
    row["id_discharge"] = safe_str(row.get("id_discharge"))
    row["material_ids"] = safe_str(row.get("material_ids"))

    # Elements and chemsys fallback
    fallback_formula = (
        row.get("framework_formula")
        or row.get("formula_discharge")
        or row.get("formula_charge")
        or row.get("battery_formula")
    )
    elements = normalize_elements_value(row.get("elements"), fallback_formula=fallback_formula)
    row["elements"] = "|".join(elements)

    if not row.get("chemsys"):
        row["chemsys"] = derive_chemsys(elements)

    return row

core_rows = []

for ion in WORKING_IONS:
    docs = raw_builtin_by_ion.get(ion, [])
    for i, doc in enumerate(docs):
        core_rows.append(harmonize_one_doc(doc, queried_ion=ion, i=i))

core_df = pd.DataFrame(core_rows)

# Save harmonized core immediately
core_path = PROCESSED_DIR / "01_multion_insertion_electrodes_core.csv"
core_df.to_csv(core_path, index=False)

print(f"Harmonized core dataset saved: {core_path}")
print(f"Core shape: {core_df.shape}")
display(core_df.head())

save_event_log()


-

## Cell 10 — Canonical IDs and group labels

In [ ]:
# ============================================================
# Add canonical IDs and group labels for future split feasibility
# ============================================================

def normalize_formula_string(s) -> str:
    s = safe_str(s).strip()
    if not s:
        return ""
    return re.sub(r"\s+", "", s)

def remove_working_ion_from_elements(elements_string: str, working_ion: str) -> list[str]:
    elements = [e for e in safe_str(elements_string).split("|") if e]
    return sorted([e for e in set(elements) if e != working_ion])

def make_electrode_uid(row) -> str:
    parts = [
        safe_str(row.get("working_ion")),
        safe_str(row.get("battery_formula")),
        safe_str(row.get("framework_formula")),
        safe_str(row.get("formula_charge")),
        safe_str(row.get("formula_discharge")),
        safe_str(row.get("id_charge")),
        safe_str(row.get("id_discharge")),
        safe_str(row.get("fracA_charge")),
        safe_str(row.get("fracA_discharge")),
        safe_str(row.get("num_steps")),
    ]
    raw = " | ".join(parts)
    return f"el_{stable_hash_text(raw, 16)}"

def make_framework_uid(row) -> str:
    wf = safe_str(row.get("working_ion"))
    fw = safe_str(row.get("framework_formula_reduced")) or safe_str(row.get("framework_formula"))
    if fw:
        return f"{wf}|{fw}"
    fallback = " | ".join([
        safe_str(row.get("framework_formula")),
        safe_str(row.get("formula_charge")),
        safe_str(row.get("formula_discharge")),
        safe_str(row.get("id_charge")),
        safe_str(row.get("id_discharge")),
    ])
    return f"{wf}|unknown_fw_{stable_hash_text(fallback, 12)}"

core_grouped_df = core_df.copy()

core_grouped_df["framework_formula_reduced"] = core_grouped_df["framework_formula"].apply(formula_to_reduced_formula)

# If framework formula is missing, fallback to charged/discharged formulas.
missing_fw = core_grouped_df["framework_formula_reduced"].astype(str).str.len() == 0
core_grouped_df.loc[missing_fw, "framework_formula_reduced"] = core_grouped_df.loc[missing_fw, "formula_discharge"].apply(formula_to_reduced_formula)

core_grouped_df["host_elements_no_working_ion"] = core_grouped_df.apply(
    lambda r: "|".join(remove_working_ion_from_elements(r.get("elements", ""), r.get("working_ion", ""))),
    axis=1,
)

core_grouped_df["host_chemsys_no_working_ion"] = core_grouped_df["host_elements_no_working_ion"].apply(
    lambda s: "-".join([x for x in safe_str(s).split("|") if x])
)

core_grouped_df["chemical_system_uid"] = core_grouped_df["chemsys"].apply(lambda x: safe_str(x))
core_grouped_df["working_ion_group"] = core_grouped_df["working_ion"].apply(lambda x: safe_str(x))

core_grouped_df["electrode_uid"] = core_grouped_df.apply(make_electrode_uid, axis=1)
core_grouped_df["framework_uid"] = core_grouped_df.apply(make_framework_uid, axis=1)

grouped_path = PROCESSED_DIR / "01_multion_insertion_electrodes_core_with_groups.csv"
core_grouped_df.to_csv(grouped_path, index=False)

print(f"Core dataset with group labels saved: {grouped_path}")
display(core_grouped_df[[
    "record_index",
    "working_ion",
    "battery_formula",
    "framework_formula",
    "framework_formula_reduced",
    "chemsys",
    "host_chemsys_no_working_ion",
    "electrode_uid",
    "framework_uid",
]].head())

save_event_log()


-

## Cell 11 — Target completeness audit

In [ ]:
# ============================================================
# Target completeness audit
# ============================================================

TARGET_COLUMNS = [
    "average_voltage",
    "capacity_grav",
    "capacity_vol",
    "energy_grav",
    "energy_vol",
    "max_delta_volume",
    "stability_charge",
    "stability_discharge",
    "stability_worst",
]

coverage_rows = []

for ion, sub in core_grouped_df.groupby("working_ion", dropna=False):
    n = len(sub)
    for target in TARGET_COLUMNS:
        if target in sub.columns:
            non_missing = pd.to_numeric(sub[target], errors="coerce").notna().sum()
            coverage_pct = 100.0 * non_missing / n if n else 0.0
        else:
            non_missing = 0
            coverage_pct = 0.0

        coverage_rows.append({
            "working_ion": ion,
            "target": target,
            "n_records": n,
            "n_non_missing": int(non_missing),
            "coverage_pct": coverage_pct,
        })

target_coverage_df = pd.DataFrame(coverage_rows)
target_coverage_df.to_csv(AUDIT_DIR / "01_target_coverage_by_working_ion.csv", index=False)

missingness_matrix = core_grouped_df[["record_index", "working_ion"] + TARGET_COLUMNS].copy()
for col in TARGET_COLUMNS:
    missingness_matrix[f"{col}_is_missing"] = pd.to_numeric(missingness_matrix[col], errors="coerce").isna()

missingness_matrix.to_csv(AUDIT_DIR / "01_target_missingness_matrix.csv", index=False)

display(target_coverage_df.pivot(index="target", columns="working_ion", values="coverage_pct").round(1))
save_event_log()


-

## Cell 12 — Record counts, framework groups, chemical-system groups

In [ ]:
# ============================================================
# Record counts and group-count audit
# ============================================================

count_rows = []

for ion, sub in core_grouped_df.groupby("working_ion", dropna=False):
    count_rows.append({
        "working_ion": ion,
        "n_records": len(sub),
        "n_unique_electrode_uid": sub["electrode_uid"].nunique(),
        "n_unique_framework_uid": sub["framework_uid"].nunique(),
        "n_unique_framework_formula_reduced": sub["framework_formula_reduced"].replace("", np.nan).nunique(dropna=True),
        "n_unique_chemsys": sub["chemsys"].replace("", np.nan).nunique(dropna=True),
        "n_unique_host_chemsys_no_working_ion": sub["host_chemsys_no_working_ion"].replace("", np.nan).nunique(dropna=True),
        "median_records_per_framework_uid": sub.groupby("framework_uid").size().median() if len(sub) else np.nan,
        "max_records_per_framework_uid": sub.groupby("framework_uid").size().max() if len(sub) else np.nan,
        "n_single_step_records": int((pd.to_numeric(sub["num_steps"], errors="coerce") == 1).sum()) if "num_steps" in sub else np.nan,
        "n_multi_step_records": int((pd.to_numeric(sub["num_steps"], errors="coerce") > 1).sum()) if "num_steps" in sub else np.nan,
    })

record_counts_df = pd.DataFrame(count_rows)
record_counts_df.to_csv(AUDIT_DIR / "01_record_counts_by_ion.csv", index=False)

# Overall group counts
overall_group_counts = pd.DataFrame([{
    "n_total_records": len(core_grouped_df),
    "n_working_ions": core_grouped_df["working_ion"].nunique(),
    "n_unique_electrode_uid": core_grouped_df["electrode_uid"].nunique(),
    "n_unique_framework_uid": core_grouped_df["framework_uid"].nunique(),
    "n_unique_framework_formula_reduced": core_grouped_df["framework_formula_reduced"].replace("", np.nan).nunique(dropna=True),
    "n_unique_chemsys": core_grouped_df["chemsys"].replace("", np.nan).nunique(dropna=True),
    "n_unique_host_chemsys_no_working_ion": core_grouped_df["host_chemsys_no_working_ion"].replace("", np.nan).nunique(dropna=True),
}])

overall_group_counts.to_csv(AUDIT_DIR / "01_overall_group_counts.csv", index=False)

display(record_counts_df)
display(overall_group_counts)
save_event_log()


-

## Cell 13 — Duplicate and near-duplicate audit

In [ ]:
# ============================================================
# Duplicate and near-duplicate audit
# ============================================================

duplicate_criteria = {
    "exact_electrode_uid": ["electrode_uid"],
    "same_ion_framework_charge_discharge_formula": [
        "working_ion",
        "framework_formula",
        "formula_charge",
        "formula_discharge",
    ],
    "same_ion_charge_discharge_ids": [
        "working_ion",
        "id_charge",
        "id_discharge",
    ],
    "same_framework_uid": ["framework_uid"],
    "same_ion_battery_formula_framework": [
        "working_ion",
        "battery_formula",
        "framework_formula",
    ],
}

duplicate_summary_rows = []
record_duplicate_flags = core_grouped_df[["record_index", "working_ion", "electrode_uid"]].copy()

for crit_name, cols in duplicate_criteria.items():
    available_cols = [c for c in cols if c in core_grouped_df.columns]
    if len(available_cols) != len(cols):
        duplicate_summary_rows.append({
            "criterion": crit_name,
            "columns": "|".join(cols),
            "status": "skipped_missing_columns",
            "n_records_in_duplicate_groups": np.nan,
            "n_duplicate_groups": np.nan,
            "duplicate_record_pct": np.nan,
        })
        continue

    temp = core_grouped_df.copy()

    # Treat empty strings as missing; avoid counting all empty IDs as one duplicate group.
    for c in available_cols:
        temp[c] = temp[c].replace("", np.nan)

    usable = temp.dropna(subset=available_cols)
    if usable.empty:
        duplicate_summary_rows.append({
            "criterion": crit_name,
            "columns": "|".join(cols),
            "status": "no_usable_rows",
            "n_records_in_duplicate_groups": 0,
            "n_duplicate_groups": 0,
            "duplicate_record_pct": 0.0,
        })
        record_duplicate_flags[f"dup_{crit_name}"] = False
        continue

    group_sizes = usable.groupby(available_cols, dropna=False).size().reset_index(name="group_size")
    duplicate_groups = group_sizes[group_sizes["group_size"] > 1]
    dup_keys = duplicate_groups[available_cols]

    marker = usable.merge(dup_keys, on=available_cols, how="inner")[["record_index"]].drop_duplicates()
    dup_record_indices = set(marker["record_index"])

    record_duplicate_flags[f"dup_{crit_name}"] = record_duplicate_flags["record_index"].isin(dup_record_indices)

    duplicate_summary_rows.append({
        "criterion": crit_name,
        "columns": "|".join(cols),
        "status": "computed",
        "n_usable_records": len(usable),
        "n_records_in_duplicate_groups": len(dup_record_indices),
        "n_duplicate_groups": len(duplicate_groups),
        "duplicate_record_pct": 100.0 * len(dup_record_indices) / len(core_grouped_df) if len(core_grouped_df) else 0.0,
    })

duplicate_summary_df = pd.DataFrame(duplicate_summary_rows)
duplicate_summary_df.to_csv(AUDIT_DIR / "01_duplicate_electrode_audit.csv", index=False)
record_duplicate_flags.to_csv(AUDIT_DIR / "01_duplicate_record_flags.csv", index=False)

display(duplicate_summary_df)
save_event_log()


-

## Cell 14 — Coarse family classification audit

In [ ]:
# ============================================================
# Coarse family classification audit
# This is only for split-feasibility assessment, not screening.
# ============================================================

TRANSITION_METALS = {
    "Sc", "Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu", "Zn",
    "Y", "Zr", "Nb", "Mo", "Tc", "Ru", "Rh", "Pd", "Ag", "Cd",
    "Hf", "Ta", "W", "Re", "Os", "Ir", "Pt", "Au", "Hg",
}

def parse_elements_from_row(row) -> set[str]:
    els = set([e for e in safe_str(row.get("elements", "")).split("|") if e])
    if els:
        return els

    for col in ["framework_formula", "formula_discharge", "formula_charge", "battery_formula"]:
        els2 = formula_to_elements(row.get(col))
        if els2:
            return set(els2)

    return set()

def classify_coarse_family(row) -> str:
    els = parse_elements_from_row(row)
    formula_text = " ".join([
        safe_str(row.get("framework_formula")),
        safe_str(row.get("formula_discharge")),
        safe_str(row.get("formula_charge")),
        safe_str(row.get("battery_formula")),
    ])

    has_O = "O" in els
    has_P = "P" in els
    has_S = "S" in els
    has_Si = "Si" in els
    has_F = "F" in els
    has_C = "C" in els
    has_N = "N" in els
    has_halide = any(x in els for x in ["F", "Cl", "Br", "I"])
    has_tm = any(x in TRANSITION_METALS for x in els)

    if has_C and not has_tm and not has_P and not has_Si:
        return "organic_or_carbon_like"

    if has_P and has_O:
        return "polyanion_phosphate_like"

    if has_S and has_O:
        return "sulfate_or_sulfur_oxoanion_like"

    if has_Si and has_O:
        return "silicate_like"

    if has_halide and not has_O:
        return "halide_or_fluoride_like"

    if has_F and has_O:
        return "oxyfluoride_or_fluoropolyanion_like"

    if has_O and has_tm:
        return "transition_metal_oxide_like"

    if has_O:
        return "oxide_like_other"

    if "S" in els and not has_O:
        return "sulfide_like"

    return "other_or_unclear"

core_grouped_df["coarse_family"] = core_grouped_df.apply(classify_coarse_family, axis=1)

family_counts_df = (
    core_grouped_df
    .groupby(["working_ion", "coarse_family"], dropna=False)
    .size()
    .reset_index(name="n_records")
    .sort_values(["working_ion", "n_records"], ascending=[True, False])
)

family_counts_df.to_csv(AUDIT_DIR / "01_family_counts_by_ion.csv", index=False)

# Save updated grouped core with family labels
core_grouped_df.to_csv(PROCESSED_DIR / "01_multion_insertion_electrodes_core_with_groups.csv", index=False)

display(family_counts_df)
save_event_log()


In [ ]:
# ============================================================
# HOTFIX before Cell 15:
# Materials Project IDs may be numeric OR alphanumeric.
#
# Old broken pattern:
#   r"\bmp-\d+\b"
#
# Current observed IDs:
#   mp-aaaaccyz
#   mp-aaaclgav
# ============================================================

MP_ID_REGEX = re.compile(r"\bmp-[A-Za-z0-9]+\b")

def extract_mp_ids_from_object(obj) -> list[str]:
    """
    Extract Materials Project IDs from any nested object.

    Handles both:
      - old numeric IDs: mp-1234
      - current alphanumeric IDs: mp-aaaaccyz
    """
    try:
        text = json.dumps(to_builtin(obj), ensure_ascii=False)
    except Exception:
        text = str(obj)

    return sorted(set(MP_ID_REGEX.findall(text)))

print("HOTFIX applied: MP_ID_REGEX now supports alphanumeric MP IDs.")

-

## Cell 15 — Linked material ID extraction

In [ ]:
# ============================================================
# Cell 15 — Linked material ID extraction
# FIXED VERSION
#
# This version extracts linked MP material IDs from:
#   1. harmonized core columns: id_charge, id_discharge, material_ids
#   2. raw nested electrode records
#
# It supports both old numeric IDs and current alphanumeric IDs.
# ============================================================

LINKED_ID_ALIAS_GROUPS = {
    "id_charge": ["id_charge", "idCharge", "charge_id", "charged_material_id"],
    "id_discharge": ["id_discharge", "idDischarge", "discharge_id", "discharged_material_id"],
    "material_ids": ["material_ids", "materialIds", "materials_ids", "material_id_list"],
    "record_source_id": ["record_source_id", "battery_id", "id", "task_id", "material_id"],
}

linked_rows = []

def append_linked_id_rows(
    rows_list,
    working_ion,
    record_index,
    electrode_uid,
    id_role,
    value,
    source,
):
    ids = extract_mp_ids_from_object(value)
    for mid in ids:
        rows_list.append({
            "working_ion": working_ion,
            "record_index": record_index,
            "electrode_uid": electrode_uid,
            "id_role": id_role,
            "material_id": mid,
            "source": source,
        })

# -
# A. Extract from harmonized core table first.
# This is the most important repair because your core table already
# contains id_charge/id_discharge/material_ids.
# -
core_id_columns = [
    "id_charge",
    "id_discharge",
    "material_ids",
]

for _, row in core_grouped_df.iterrows():
    ion = safe_str(row.get("working_ion"))
    record_index = safe_str(row.get("record_index"))
    electrode_uid = safe_str(row.get("electrode_uid"))

    for col in core_id_columns:
        if col in core_grouped_df.columns:
            append_linked_id_rows(
                rows_list=linked_rows,
                working_ion=ion,
                record_index=record_index,
                electrode_uid=electrode_uid,
                id_role=col,
                value=row.get(col),
                source="harmonized_core_column",
            )

# -
# B. Also extract from raw nested electrode records.
# This keeps the original audit behavior.
# -
for ion in WORKING_IONS:
    docs = raw_builtin_by_ion.get(ion, [])

    sub_core = (
        core_grouped_df[core_grouped_df["source_queried_working_ion"] == ion]
        .reset_index(drop=True)
        if "source_queried_working_ion" in core_grouped_df.columns
        else core_grouped_df[core_grouped_df["working_ion"] == ion].reset_index(drop=True)
    )

    for i, doc in enumerate(docs):
        if i < len(sub_core):
            record_index = safe_str(sub_core.loc[i, "record_index"])
            electrode_uid = safe_str(sub_core.loc[i, "electrode_uid"])
        else:
            record_index = f"{ion}_{i:06d}"
            electrode_uid = ""

        # Role-specific aliases
        for role, aliases in LINKED_ID_ALIAS_GROUPS.items():
            val = get_any_alias(doc, aliases, default=None)
            append_linked_id_rows(
                rows_list=linked_rows,
                working_ion=ion,
                record_index=record_index,
                electrode_uid=electrode_uid,
                id_role=role,
                value=val,
                source="raw_role_specific_alias",
            )

        # Whole-record regex fallback
        append_linked_id_rows(
            rows_list=linked_rows,
            working_ion=ion,
            record_index=record_index,
            electrode_uid=electrode_uid,
            id_role="anywhere_raw_record",
            value=doc,
            source="whole_raw_record_regex",
        )

# -
# C. Save long linked-ID table
# -
linked_ids_long_df = pd.DataFrame(linked_rows)

if linked_ids_long_df.empty:
    linked_ids_long_df = pd.DataFrame(columns=[
        "working_ion",
        "record_index",
        "electrode_uid",
        "id_role",
        "material_id",
        "source",
    ])
    log_event(
        "linked_id_extraction",
        "WARNING",
        "No linked MP material IDs extracted after alphanumeric MP-ID hotfix.",
    )
else:
    linked_ids_long_df = linked_ids_long_df.drop_duplicates()

linked_ids_long_path = PROCESSED_DIR / "01_linked_material_ids_long.csv"
linked_ids_long_df.to_csv(linked_ids_long_path, index=False)

# -
# D. Summary table
# -
if linked_ids_long_df.empty:
    linked_id_summary_df = pd.DataFrame(columns=[
        "working_ion",
        "id_role",
        "source",
        "n_unique_material_ids",
        "n_rows",
    ])
else:
    linked_id_summary_df = (
        linked_ids_long_df
        .groupby(["working_ion", "id_role", "source"], dropna=False)
        .agg(
            n_unique_material_ids=("material_id", "nunique"),
            n_rows=("material_id", "count"),
        )
        .reset_index()
        .sort_values(["working_ion", "id_role", "source"])
    )

linked_id_summary_path = AUDIT_DIR / "01_linked_material_ids_summary.csv"
linked_id_summary_df.to_csv(linked_id_summary_path, index=False)

# -
# E. Print sanity check
# -
n_unique_linked_ids = (
    linked_ids_long_df["material_id"].nunique()
    if not linked_ids_long_df.empty
    else 0
)

print(f"Linked MP material IDs extracted: {n_unique_linked_ids}")
print(f"Saved: {linked_ids_long_path}")
print(f"Saved: {linked_id_summary_path}")

if n_unique_linked_ids == 0:
    print("WARNING: linked ID extraction is still empty. Check id_charge/id_discharge/material_ids columns.")
else:
    display(linked_ids_long_df.head(20))
    display(linked_id_summary_df)

save_event_log()

-

## Cell 16 — Query linked MP summary availability

In [ ]:
# ============================================================
# Linked Materials Project summary availability audit
# ============================================================

SUMMARY_FIELD_CANDIDATES = [
    "material_id",
    "formula_pretty",
    "composition_reduced",
    "chemsys",
    "elements",
    "nsites",
    "volume",
    "density",
    "density_atomic",
    "symmetry",
    "energy_above_hull",
    "formation_energy_per_atom",
    "is_stable",
    "theoretical",
    "band_gap",
    "is_metal",
    "last_updated",
]

def select_summary_fields(available_fields: list[str]) -> list[str] | None:
    if available_fields:
        selected = [f for f in SUMMARY_FIELD_CANDIDATES if f in set(available_fields)]
        return selected if selected else None
    return SUMMARY_FIELD_CANDIDATES

def query_summary_docs(material_ids: list[str], fields: list[str] | None, chunk_size: int = 500):
    if summary_rester is None:
        log_event("summary_query", "WARNING", "Summary rester unavailable; skipping linked summary query.")
        return [], [{"status": "summary_rester_unavailable"}]

    all_docs = []
    query_logs = []

    for chunk in tqdm(list(chunked(material_ids, chunk_size)), desc="Querying linked MP summaries"):
        variants = []

        if fields is not None:
            variants.append({"material_ids": chunk, "fields": fields})
        variants.append({"material_ids": chunk})

        success = False
        for kwargs in variants:
            docs, err = call_search(summary_rester, kwargs, stage="linked_summary_query")
            query_logs.append({
                "chunk_size": len(chunk),
                "kwargs_keys": list(kwargs.keys()),
                "success": docs is not None,
                "n_docs": len(docs) if docs is not None else None,
                "error": err,
            })
            if docs is not None:
                all_docs.extend(docs)
                success = True
                break

        if not success:
            log_event("summary_query", "WARNING", "All summary query variants failed for a chunk.", {"chunk_size": len(chunk)})

    return all_docs, query_logs

unique_linked_ids = sorted(linked_ids_long_df["material_id"].dropna().unique().tolist()) if not linked_ids_long_df.empty else []

summary_fields = select_summary_fields(available_summary_fields)

summary_docs, summary_query_logs = query_summary_docs(unique_linked_ids, summary_fields, chunk_size=500)

summary_builtin = [to_builtin(d) for d in summary_docs]
summary_df = docs_to_flat_dataframe(summary_builtin)

summary_path = PROCESSED_DIR / "01_linked_materials_summary.csv"
summary_df.to_csv(summary_path, index=False)

write_json_safe(summary_query_logs, METADATA_DIR / "01_linked_summary_query_log.json")

# Coverage audit
if not summary_df.empty:
    material_id_col = None
    for c in ["material_id", "materialId"]:
        if c in summary_df.columns:
            material_id_col = c
            break

    returned_ids = set(summary_df[material_id_col].astype(str)) if material_id_col else set(extract_mp_ids_from_object(summary_builtin))
else:
    returned_ids = set()

requested_ids = set(unique_linked_ids)
missing_ids = sorted(requested_ids - returned_ids)

linked_summary_coverage = pd.DataFrame([{
    "n_requested_unique_linked_material_ids": len(requested_ids),
    "n_returned_unique_summary_material_ids": len(returned_ids),
    "summary_coverage_pct": 100.0 * len(returned_ids) / len(requested_ids) if requested_ids else 0.0,
    "summary_route_selected": summary_route_label,
}])

linked_summary_coverage.to_csv(AUDIT_DIR / "01_linked_summary_coverage_audit.csv", index=False)

pd.DataFrame({"missing_material_id": missing_ids}).to_csv(
    AUDIT_DIR / "01_linked_summary_missing_ids.csv",
    index=False,
)

display(linked_summary_coverage)
print(f"Linked summary saved: {summary_path}")
save_event_log()


-

## Cell 17 — Structure availability audit

In [ ]:
# ============================================================
# Structure availability audit
# No descriptor generation is performed here.
# ============================================================

def has_structure_like_value(obj, max_depth: int = 20) -> bool:
    """
    Recursively detect whether a raw electrode document appears to contain
    a structure-like object. This does not validate it as a pymatgen Structure.
    """
    if max_depth <= 0:
        return False

    if isinstance(obj, dict):
        for k, v in obj.items():
            k_lower = str(k).lower()
            if "structure" in k_lower and not is_missing_like(v):
                return True
            if has_structure_like_value(v, max_depth=max_depth - 1):
                return True

    elif isinstance(obj, list):
        for item in obj:
            if has_structure_like_value(item, max_depth=max_depth - 1):
                return True

    return False

raw_structure_rows = []

for ion in WORKING_IONS:
    docs = raw_builtin_by_ion.get(ion, [])
    sub_core = core_grouped_df[core_grouped_df["source_queried_working_ion"] == ion].reset_index(drop=True)

    for i, doc in enumerate(docs):
        record_index = sub_core.loc[i, "record_index"] if i < len(sub_core) else f"{ion}_{i:06d}"
        electrode_uid = sub_core.loc[i, "electrode_uid"] if i < len(sub_core) else ""
        raw_structure_rows.append({
            "working_ion": ion,
            "record_index": record_index,
            "electrode_uid": electrode_uid,
            "raw_electrode_has_structure_like_field": has_structure_like_value(doc),
        })

raw_structure_df = pd.DataFrame(raw_structure_rows)

# Query linked structures from summary endpoint if possible.
structure_summary_rows = []
structure_query_log = []

if summary_rester is not None and unique_linked_ids:
    structure_fields = None
    if available_summary_fields:
        if "structure" in set(available_summary_fields) and "material_id" in set(available_summary_fields):
            structure_fields = ["material_id", "structure"]
        elif "structure" in set(available_summary_fields):
            structure_fields = ["structure"]

    # If field discovery is unavailable, try candidate fields and fall back internally.
    if not available_summary_fields:
        structure_fields = ["material_id", "structure"]

    for chunk in tqdm(list(chunked(unique_linked_ids, 100)), desc="Checking linked structure availability"):
        variants = []
        if structure_fields is not None:
            variants.append({"material_ids": chunk, "fields": structure_fields})
        variants.append({"material_ids": chunk})

        got_docs = None
        for kwargs in variants:
            docs, err = call_search(summary_rester, kwargs, stage="linked_structure_query")
            structure_query_log.append({
                "chunk_size": len(chunk),
                "kwargs_keys": list(kwargs.keys()),
                "success": docs is not None,
                "n_docs": len(docs) if docs is not None else None,
                "error": err,
            })
            if docs is not None:
                got_docs = [to_builtin(d) for d in docs]
                break

        if got_docs is None:
            continue

        for d in got_docs:
            mid = get_any_alias(d, ["material_id", "materialId"], default=None)
            if is_missing_like(mid):
                ids = extract_mp_ids_from_object(d)
                mid = ids[0] if ids else ""

            structure_obj = get_any_alias(d, ["structure"], default=None)
            has_structure = not is_missing_like(structure_obj)

            nsites = np.nan
            structure_formula = ""

            if has_structure:
                if isinstance(structure_obj, dict):
                    if "sites" in structure_obj and isinstance(structure_obj["sites"], list):
                        nsites = len(structure_obj["sites"])
                    structure_formula = safe_str(
                        structure_obj.get("formula", "")
                        or structure_obj.get("composition", "")
                    )
                else:
                    # If object string/dict conversion is unavailable, only mark presence.
                    structure_formula = ""

            structure_summary_rows.append({
                "material_id": safe_str(mid),
                "linked_summary_has_structure": bool(has_structure),
                "structure_nsites_if_parseable": nsites,
                "structure_formula_if_available": structure_formula,
            })

else:
    log_event("structure_audit", "WARNING", "Summary rester or linked IDs unavailable; linked structure query skipped.")

linked_structure_df = pd.DataFrame(structure_summary_rows)

if linked_structure_df.empty:
    linked_structure_df = pd.DataFrame(columns=[
        "material_id",
        "linked_summary_has_structure",
        "structure_nsites_if_parseable",
        "structure_formula_if_available",
    ])

linked_structure_df = linked_structure_df.drop_duplicates(subset=["material_id"])

linked_structure_df.to_csv(AUDIT_DIR / "01_linked_structure_availability_by_material_id.csv", index=False)
write_json_safe(structure_query_log, METADATA_DIR / "01_linked_structure_query_log.json")

# Merge linked structure availability back to electrode records.
if not linked_ids_long_df.empty and not linked_structure_df.empty:
    linked_with_structure = linked_ids_long_df.merge(
        linked_structure_df[["material_id", "linked_summary_has_structure"]],
        on="material_id",
        how="left",
    )
    electrode_linked_structure = (
        linked_with_structure
        .groupby(["record_index", "electrode_uid"], dropna=False)["linked_summary_has_structure"]
        .max()
        .reset_index()
        .rename(columns={"linked_summary_has_structure": "any_linked_material_has_structure"})
    )
else:
    electrode_linked_structure = pd.DataFrame(columns=[
        "record_index", "electrode_uid", "any_linked_material_has_structure"
    ])

structure_record_df = raw_structure_df.merge(
    electrode_linked_structure,
    on=["record_index", "electrode_uid"],
    how="left",
)

structure_record_df["any_linked_material_has_structure"] = structure_record_df[
    "any_linked_material_has_structure"
].fillna(False).astype(bool)

structure_record_df["record_has_any_structure_route"] = (
    structure_record_df["raw_electrode_has_structure_like_field"].astype(bool)
    | structure_record_df["any_linked_material_has_structure"].astype(bool)
)

structure_record_df.to_csv(AUDIT_DIR / "01_structure_availability_by_record.csv", index=False)

structure_by_ion_df = (
    structure_record_df
    .groupby("working_ion", dropna=False)
    .agg(
        n_records=("record_index", "count"),
        n_raw_structure_like=("raw_electrode_has_structure_like_field", "sum"),
        n_linked_structure=("any_linked_material_has_structure", "sum"),
        n_any_structure_route=("record_has_any_structure_route", "sum"),
    )
    .reset_index()
)

for col in ["n_raw_structure_like", "n_linked_structure", "n_any_structure_route"]:
    structure_by_ion_df[col.replace("n_", "pct_")] = (
        100.0 * structure_by_ion_df[col] / structure_by_ion_df["n_records"].replace(0, np.nan)
    )

structure_by_ion_df.to_csv(AUDIT_DIR / "01_structure_availability_by_ion.csv", index=False)

display(structure_by_ion_df)
save_event_log()


-

## Cell 18 — Cross-ion framework, chemical-system, and formula overlap

In [ ]:
# ============================================================
# Cross-ion overlap audits
# ============================================================

def cross_ion_overlap_table(df: pd.DataFrame, key_col: str, output_name: str):
    rows = []
    sets_by_ion = {}

    for ion in WORKING_IONS:
        vals = (
            df.loc[df["working_ion"] == ion, key_col]
            .replace("", np.nan)
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )
        sets_by_ion[ion] = set(vals)

    for ion_a, ion_b in combinations(WORKING_IONS, 2):
        set_a = sets_by_ion.get(ion_a, set())
        set_b = sets_by_ion.get(ion_b, set())
        inter = set_a & set_b
        union = set_a | set_b
        rows.append({
            "key_col": key_col,
            "ion_a": ion_a,
            "ion_b": ion_b,
            "n_a": len(set_a),
            "n_b": len(set_b),
            "n_intersection": len(inter),
            "n_union": len(union),
            "jaccard_overlap": len(inter) / len(union) if union else 0.0,
            "intersection_values_sample": "|".join(sorted(list(inter))[:50]),
        })

    out = pd.DataFrame(rows)
    out.to_csv(AUDIT_DIR / output_name, index=False)
    return out

framework_overlap_df = cross_ion_overlap_table(
    core_grouped_df,
    key_col="framework_formula_reduced",
    output_name="01_cross_ion_framework_overlap.csv",
)

chemsys_overlap_df = cross_ion_overlap_table(
    core_grouped_df,
    key_col="host_chemsys_no_working_ion",
    output_name="01_cross_ion_chemsys_overlap.csv",
)

formula_overlap_df = cross_ion_overlap_table(
    core_grouped_df,
    key_col="battery_formula",
    output_name="01_cross_ion_formula_overlap.csv",
)

print("Framework overlap:")
display(framework_overlap_df)

print("Host chemical-system overlap excluding working ion:")
display(chemsys_overlap_df)

print("Battery-formula overlap:")
display(formula_overlap_df)

save_event_log()


-

## Cell 19 — Dataset feasibility gate

In [ ]:
# ============================================================
# Final dataset-feasibility gate
# ============================================================

def gate_status_count(observed, pass_min, conditional_min):
    if observed >= pass_min:
        return "PASS"
    if observed >= conditional_min:
        return "CONDITIONAL_PASS"
    return "FAIL"

def gate_status_pct(observed_pct, pass_min_pct, conditional_min_pct):
    if observed_pct >= pass_min_pct:
        return "PASS"
    if observed_pct >= conditional_min_pct:
        return "CONDITIONAL_PASS"
    return "FAIL"

def status_rank(status):
    return {"PASS": 0, "CONDITIONAL_PASS": 1, "FAIL": 2}.get(status, 2)

gate_rows = []

# -
# Record-count gates
# -
record_count_lookup = dict(zip(record_counts_df["working_ion"], record_counts_df["n_records"]))

record_thresholds = {
    "Li": {"pass_min": 500, "conditional_min": 250},
    "Na": {"pass_min": 300, "conditional_min": 200},
    "K": {"pass_min": 75, "conditional_min": 50},
}

for ion in WORKING_IONS:
    observed = int(record_count_lookup.get(ion, 0))
    th = record_thresholds[ion]
    status = gate_status_count(observed, th["pass_min"], th["conditional_min"])
    gate_rows.append({
        "gate_item": f"{ion}_record_count",
        "status": status,
        "observed_value": observed,
        "minimum_required_for_pass": th["pass_min"],
        "minimum_required_for_conditional": th["conditional_min"],
        "critical_for_path_a": True,
        "interpretation": f"{ion} insertion-electrode record count.",
        "action": "Proceed if PASS; weaken benchmark claims if CONDITIONAL_PASS; redesign if FAIL.",
    })

# -
# Target-completeness gates
# -
coverage_pivot = target_coverage_df.pivot(index="target", columns="working_ion", values="coverage_pct")

main_targets = ["average_voltage", "capacity_grav", "energy_grav"]
secondary_targets = ["capacity_vol", "energy_vol", "max_delta_volume", "stability_charge", "stability_discharge", "stability_worst"]

for target in main_targets:
    vals = []
    for ion in WORKING_IONS:
        try:
            vals.append(float(coverage_pivot.loc[target, ion]))
        except Exception:
            vals.append(0.0)
    observed_min = min(vals) if vals else 0.0
    status = gate_status_pct(observed_min, pass_min_pct=90.0, conditional_min_pct=75.0)
    gate_rows.append({
        "gate_item": f"{target}_minimum_coverage_across_Li_Na_K",
        "status": status,
        "observed_value": observed_min,
        "minimum_required_for_pass": 90.0,
        "minimum_required_for_conditional": 75.0,
        "critical_for_path_a": True,
        "interpretation": f"Minimum {target} completeness across Li, Na, K.",
        "action": "Main ML benchmark target only if PASS or CONDITIONAL_PASS.",
    })

for target in secondary_targets:
    vals = []
    for ion in WORKING_IONS:
        try:
            vals.append(float(coverage_pivot.loc[target, ion]))
        except Exception:
            vals.append(0.0)
    observed_min = min(vals) if vals else 0.0
    status = gate_status_pct(observed_min, pass_min_pct=70.0, conditional_min_pct=50.0)
    gate_rows.append({
        "gate_item": f"{target}_minimum_coverage_across_Li_Na_K",
        "status": status,
        "observed_value": observed_min,
        "minimum_required_for_pass": 70.0,
        "minimum_required_for_conditional": 50.0,
        "critical_for_path_a": False,
        "interpretation": f"Minimum {target} completeness across Li, Na, K.",
        "action": "Use as secondary target only if sufficiently complete.",
    })

# -
# Group-count gates
# -
n_total_records = len(core_grouped_df)
n_framework_groups = core_grouped_df["framework_uid"].replace("", np.nan).nunique(dropna=True)
n_framework_formula_groups = core_grouped_df["framework_formula_reduced"].replace("", np.nan).nunique(dropna=True)
n_chemsys_groups = core_grouped_df["chemsys"].replace("", np.nan).nunique(dropna=True)
n_host_chemsys_groups = core_grouped_df["host_chemsys_no_working_ion"].replace("", np.nan).nunique(dropna=True)
n_families = core_grouped_df["coarse_family"].replace("", np.nan).nunique(dropna=True)

group_gate_specs = [
    ("total_records", n_total_records, 875, 500, True, "Total Li-Na-K records."),
    ("framework_uid_groups", n_framework_groups, 300, 150, True, "Framework groups for grouped validation."),
    ("framework_formula_groups", n_framework_formula_groups, 200, 100, False, "Reduced framework-formula groups."),
    ("chemical_system_groups", n_chemsys_groups, 150, 75, True, "Chemical systems for leave-chemical-system-out feasibility."),
    ("host_chemsys_no_working_ion_groups", n_host_chemsys_groups, 100, 50, False, "Host chemical systems excluding working ion."),
    ("coarse_family_groups", n_families, 5, 3, False, "Coarse family diversity for leave-family-out feasibility."),
]

for item, observed, pass_min, cond_min, critical, interp in group_gate_specs:
    status = gate_status_count(int(observed), pass_min, cond_min)
    gate_rows.append({
        "gate_item": item,
        "status": status,
        "observed_value": int(observed),
        "minimum_required_for_pass": pass_min,
        "minimum_required_for_conditional": cond_min,
        "critical_for_path_a": critical,
        "interpretation": interp,
        "action": "Proceed if PASS; inspect split design if CONDITIONAL_PASS; redesign if FAIL.",
    })

# -
# Duplicate-burden gate
# -
try:
    exact_dup_pct = float(
        duplicate_summary_df.loc[
            duplicate_summary_df["criterion"] == "exact_electrode_uid",
            "duplicate_record_pct"
        ].iloc[0]
    )
except Exception:
    exact_dup_pct = 100.0

if exact_dup_pct <= 10.0:
    dup_status = "PASS"
elif exact_dup_pct <= 30.0:
    dup_status = "CONDITIONAL_PASS"
else:
    dup_status = "FAIL"

gate_rows.append({
    "gate_item": "exact_duplicate_burden_pct",
    "status": dup_status,
    "observed_value": exact_dup_pct,
    "minimum_required_for_pass": "<=10",
    "minimum_required_for_conditional": "<=30",
    "critical_for_path_a": False,
    "interpretation": "Exact electrode UID duplicate burden.",
    "action": "Flag duplicates; do not delete aggressively before split-design audit.",
})

# -
# Linked summary and structure gates
# -
summary_coverage_pct = float(linked_summary_coverage["summary_coverage_pct"].iloc[0]) if not linked_summary_coverage.empty else 0.0
summary_status = gate_status_pct(summary_coverage_pct, 80.0, 50.0)

gate_rows.append({
    "gate_item": "linked_material_summary_coverage_pct",
    "status": summary_status,
    "observed_value": summary_coverage_pct,
    "minimum_required_for_pass": 80.0,
    "minimum_required_for_conditional": 50.0,
    "critical_for_path_a": False,
    "interpretation": "Coverage of linked MP material summaries.",
    "action": "Needed for later structure/DFT provenance and P3 support.",
})

if not structure_by_ion_df.empty:
    structure_min_pct = float(structure_by_ion_df["pct_any_structure_route"].min())
else:
    structure_min_pct = 0.0

structure_status = gate_status_pct(structure_min_pct, 70.0, 40.0)

gate_rows.append({
    "gate_item": "minimum_structure_route_coverage_pct_across_ions",
    "status": structure_status,
    "observed_value": structure_min_pct,
    "minimum_required_for_pass": 70.0,
    "minimum_required_for_conditional": 40.0,
    "critical_for_path_a": False,
    "interpretation": "Minimum per-ion availability of raw or linked structure route.",
    "action": "P2 composition+structure protocol only defensible if adequate.",
})

# -
# Cross-ion overlap gates
# -
try:
    min_framework_jaccard = float(framework_overlap_df["jaccard_overlap"].min())
    max_framework_intersection = int(framework_overlap_df["n_intersection"].max())
except Exception:
    min_framework_jaccard = 0.0
    max_framework_intersection = 0

try:
    max_host_chemsys_intersection = int(chemsys_overlap_df["n_intersection"].max())
except Exception:
    max_host_chemsys_intersection = 0

# Cross-ion overlap is useful but not strictly required; leave-working-ion-out can be harsher when overlap is low.
if max_host_chemsys_intersection >= 20:
    overlap_status = "PASS"
elif max_host_chemsys_intersection >= 5:
    overlap_status = "CONDITIONAL_PASS"
else:
    overlap_status = "FAIL"

gate_rows.append({
    "gate_item": "cross_ion_host_chemsys_overlap",
    "status": overlap_status,
    "observed_value": max_host_chemsys_intersection,
    "minimum_required_for_pass": 20,
    "minimum_required_for_conditional": 5,
    "critical_for_path_a": False,
    "interpretation": "Maximum pairwise overlap of host chemical systems excluding working ion.",
    "action": "Low overlap means leave-working-ion-out is a severe chemical-domain shift, not a symmetric interpolation test.",
})

# -
# Leave-working-ion-out feasibility gate
# -
ion_counts_ok = all(
    gate_status_count(int(record_count_lookup.get(ion, 0)), record_thresholds[ion]["pass_min"], record_thresholds[ion]["conditional_min"])
    in {"PASS", "CONDITIONAL_PASS"}
    for ion in WORKING_IONS
)

main_targets_ok = all(
    row["status"] in {"PASS", "CONDITIONAL_PASS"}
    for row in gate_rows
    if row["gate_item"] in [f"{t}_minimum_coverage_across_Li_Na_K" for t in main_targets]
)

if ion_counts_ok and main_targets_ok:
    lwoo_status = "PASS"
else:
    lwoo_status = "FAIL"

gate_rows.append({
    "gate_item": "leave_working_ion_out_feasibility",
    "status": lwoo_status,
    "observed_value": f"ion_counts_ok={ion_counts_ok}; main_targets_ok={main_targets_ok}",
    "minimum_required_for_pass": "all ions >= conditional count and main targets >= conditional coverage",
    "minimum_required_for_conditional": "not_applicable",
    "critical_for_path_a": True,
    "interpretation": "Whether leave-working-ion-out validation is statistically meaningful.",
    "action": "If FAIL, redesign Path A before Notebook 02.",
})

feasibility_gate_df = pd.DataFrame(gate_rows)

# Overall final decision
critical_gate_df = feasibility_gate_df[feasibility_gate_df["critical_for_path_a"].astype(bool)]

has_critical_fail = (critical_gate_df["status"] == "FAIL").any()
has_any_fail = (feasibility_gate_df["status"] == "FAIL").any()
has_any_conditional = (feasibility_gate_df["status"] == "CONDITIONAL_PASS").any()

if has_critical_fail:
    FINAL_DECISION = "NO_GO_REDESIGN_REQUIRED"
elif has_any_fail or has_any_conditional:
    FINAL_DECISION = "CONDITIONAL_GO_MODIFY_PATH_A"
else:
    FINAL_DECISION = "FULL_GO_TO_NOTEBOOK_09"

feasibility_gate_df["final_notebook08_decision"] = FINAL_DECISION

feasibility_gate_path = AUDIT_DIR / "01_dataset_feasibility_gate.csv"
feasibility_gate_df.to_csv(feasibility_gate_path, index=False)

write_json_safe(
    {
        "final_decision": FINAL_DECISION,
        "has_critical_fail": bool(has_critical_fail),
        "has_any_fail": bool(has_any_fail),
        "has_any_conditional": bool(has_any_conditional),
        "n_total_records": int(n_total_records),
        "records_by_ion": {ion: int(record_count_lookup.get(ion, 0)) for ion in WORKING_IONS},
        "n_framework_groups": int(n_framework_groups),
        "n_chemsys_groups": int(n_chemsys_groups),
        "minimum_structure_route_coverage_pct_across_ions": structure_min_pct,
        "summary_coverage_pct": summary_coverage_pct,
    },
    METADATA_DIR / "01_final_decision.json",
)

display(feasibility_gate_df)

print("\n" + "=" * 80)
print(f"Notebook 01 FINAL DECISION: {FINAL_DECISION}")
print("=" * 80)

save_event_log()


-

## Cell 20 — Final output manifest

In [ ]:
# ============================================================
# Final output manifest
# ============================================================

def list_output_files(base_dir: Path):
    rows = []
    for path in sorted(base_dir.rglob("*")):
        if path.is_file():
            rows.append({
                "relative_path": str(path.relative_to(base_dir)),
                "size_bytes": path.stat().st_size,
                "modified_utc": datetime.fromtimestamp(path.stat().st_mtime, timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
            })
    return pd.DataFrame(rows)

output_manifest_df = list_output_files(BASE_DIR)
output_manifest_df.to_csv(METADATA_DIR / "01_output_file_manifest.csv", index=False)

print(f"All Notebook 01 outputs saved under: {BASE_DIR}")
display(output_manifest_df)

print("\nRequired key outputs:")
for p in [
    RAW_DIR / "mp_Li_insertion_electrodes_raw.json",
    RAW_DIR / "mp_Na_insertion_electrodes_raw.json",
    RAW_DIR / "mp_K_insertion_electrodes_raw.json",
    PROCESSED_DIR / "01_multion_insertion_electrodes_core.csv",
    PROCESSED_DIR / "01_multion_insertion_electrodes_core_with_groups.csv",
    AUDIT_DIR / "01_target_coverage_by_working_ion.csv",
    AUDIT_DIR / "01_record_counts_by_ion.csv",
    AUDIT_DIR / "01_duplicate_electrode_audit.csv",
    AUDIT_DIR / "01_family_counts_by_ion.csv",
    PROCESSED_DIR / "01_linked_material_ids_long.csv",
    PROCESSED_DIR / "01_linked_materials_summary.csv",
    AUDIT_DIR / "01_structure_availability_by_ion.csv",
    AUDIT_DIR / "01_cross_ion_framework_overlap.csv",
    AUDIT_DIR / "01_cross_ion_chemsys_overlap.csv",
    AUDIT_DIR / "01_dataset_feasibility_gate.csv",
    METADATA_DIR / "01_final_decision.json",
]:
    print(f" - {p}  {'[OK]' if p.exists() else '[MISSING]'}")

print("\nNotebook 08 complete.")
print(f"Final decision: {FINAL_DECISION}")


-

After running this notebook, the next thing to inspect is:

```text
the canonical Notebook 01 data and provenance directories/audit/01_dataset_feasibility_gate.csv
```

Do **not** move to Notebook 02 until that file gives either `FULL_GO_TO_NOTEBOOK_09` or a defensible `CONDITIONAL_GO_MODIFY_PATH_A`.

[1]: https://docs.materialsproject.org/downloading-data/using-the-api/getting-started?utm_source=chatgpt.com "Getting Started"